# 14 - Validate Packed Ternary Reconstruction

Esta etapa valida a prova prática que faltava depois do `scaled500`:

1. Ler `deploy/manifest.json` e `deploy/mlp_ternary_2bit.bin`.
2. Validar integridade estrutural dos 78 MLPs empacotados em 2-bit.
3. Reconstruir os MLPs a partir do binário packed.
4. Substituir os MLPs do `student_pruned` por esses pesos reconstruídos.
5. Aplicar o adapter `student_ternary_recovered_l78`.
6. Rodar a bateria local de 100 perguntas.

Importante: este notebook ainda **não** prova runtime eficiente direto em 2-bit. Ele prova que o pacote packed contém informação suficiente para reconstruir um modelo funcional.


**Evaluation revision:** this notebook imports rpt-exact-match-v2 from the checked-out RPT source. The 100 questions are a development/regression suite used in recovery training. Saved outputs are historical and were not rerun by this edit. For independent evaluation use validation/lm_eval_packed.py and a frozen split manifest.


In [ ]:
# Auto-bootstrap para Colab/local. Rode esta célula primeiro.
from pathlib import Path
import os, sys, json, gc, re, time, hashlib, zipfile, subprocess

IN_COLAB = 'google.colab' in sys.modules
PROJECT_NAME = 'gemma4_theory'
COLAB_DRIVE_ROOT = '/content/drive/MyDrive/RPT'
LOCAL_ROOT = str(Path.cwd().resolve())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path(COLAB_DRIVE_ROOT) / PROJECT_NAME
else:
    PROJECT_ROOT = Path(LOCAL_ROOT) / PROJECT_NAME if Path(LOCAL_ROOT).name != PROJECT_NAME else Path(LOCAL_ROOT)

DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
CHECKPOINTS_DIR = PROJECT_ROOT / 'checkpoints'
DEPLOY_DIR = PROJECT_ROOT / 'deploy'
EXPORTS_DIR = PROJECT_ROOT / 'exports'
AUDIT_DIR = RESULTS_DIR / 'deployment_audit'
for p in [DATA_DIR, RESULTS_DIR, CHECKPOINTS_DIR, DEPLOY_DIR, EXPORTS_DIR, AUDIT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('CHECKPOINTS_DIR =', CHECKPOINTS_DIR)
print('DEPLOY_DIR =', DEPLOY_DIR)
print('AUDIT_DIR =', AUDIT_DIR)


In [ ]:
# Dependências para reconstrução + avaliação local.
import subprocess, sys
subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    '-U',
    'transformers',
    'accelerate',
    'sentencepiece',
    'safetensors',
    'peft',
    'datasets',
    'torchao>=0.16.0',
])


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

HF_TOKEN = os.environ.get('HF_TOKEN')
PRUNED_CHECKPOINT = CHECKPOINTS_DIR / 'student_pruned'
L78_ADAPTER = CHECKPOINTS_DIR / 'student_ternary_recovered_l78'
QUESTIONS_PATH = DATA_DIR / 'gemma4_100_questions.json'

# Mantém o notebook autocontido no Colab: recria as 100 perguntas se o arquivo não existir no Drive.
QUESTIONS_PAYLOAD = '[\n  {\n    "id": 1,\n    "question": "Qual a capital da França?",\n    "expected_answer": "Paris",\n    "accepted_answers": [\n      "Paris"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 2,\n    "question": "Qual a capital da Alemanha?",\n    "expected_answer": "Berlim",\n    "accepted_answers": [\n      "Berlim",\n      "Berlin"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 3,\n    "question": "Qual a capital da Itália?",\n    "expected_answer": "Roma",\n    "accepted_answers": [\n      "Roma",\n      "Rome"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 4,\n    "question": "Qual a capital da Espanha?",\n    "expected_answer": "Madri",\n    "accepted_answers": [\n      "Madri",\n      "Madrid"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 5,\n    "question": "Qual a capital de Portugal?",\n    "expected_answer": "Lisboa",\n    "accepted_answers": [\n      "Lisboa"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 6,\n    "question": "Qual a capital do Japão?",\n    "expected_answer": "Tóquio",\n    "accepted_answers": [\n      "Tóquio",\n      "Toquio",\n      "Tokyo"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 7,\n    "question": "Qual a capital da China?",\n    "expected_answer": "Pequim",\n    "accepted_answers": [\n      "Pequim",\n      "Beijing",\n      "Pekin"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 8,\n    "question": "Qual a capital da Argentina?",\n    "expected_answer": "Buenos Aires",\n    "accepted_answers": [\n      "Buenos Aires"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 9,\n    "question": "Qual a capital do Canadá?",\n    "expected_answer": "Ottawa",\n    "accepted_answers": [\n      "Ottawa"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 10,\n    "question": "Qual a capital da Austrália?",\n    "expected_answer": "Camberra",\n    "accepted_answers": [\n      "Camberra",\n      "Canberra"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 11,\n    "question": "Quem pintou a Mona Lisa?",\n    "expected_answer": "Leonardo da Vinci",\n    "accepted_answers": [\n      "Leonardo da Vinci",\n      "Da Vinci"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 12,\n    "question": "Quem escreveu Dom Quixote?",\n    "expected_answer": "Miguel de Cervantes",\n    "accepted_answers": [\n      "Miguel de Cervantes",\n      "Cervantes"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 13,\n    "question": "Quem escreveu Hamlet?",\n    "expected_answer": "William Shakespeare",\n    "accepted_answers": [\n      "William Shakespeare",\n      "Shakespeare"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 14,\n    "question": "Quem descobriu a penicilina?",\n    "expected_answer": "Alexander Fleming",\n    "accepted_answers": [\n      "Alexander Fleming",\n      "Fleming"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 15,\n    "question": "Quem desenvolveu a teoria da relatividade?",\n    "expected_answer": "Albert Einstein",\n    "accepted_answers": [\n      "Albert Einstein",\n      "Einstein"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 16,\n    "question": "Qual planeta é conhecido como Planeta Vermelho?",\n    "expected_answer": "Marte",\n    "accepted_answers": [\n      "Marte",\n      "Mars"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 17,\n    "question": "Qual é o maior planeta do Sistema Solar?",\n    "expected_answer": "Júpiter",\n    "accepted_answers": [\n      "Júpiter",\n      "Jupiter"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 18,\n    "question": "Quantos continentes existem tradicionalmente?",\n    "expected_answer": "7",\n    "accepted_answers": [\n      "7",\n      "sete"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 19,\n    "question": "Qual é o oceano mais extenso da Terra?",\n    "expected_answer": "Oceano Pacífico",\n    "accepted_answers": [\n      "Oceano Pacífico",\n      "Pacífico",\n      "Pacific Ocean"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 20,\n    "question": "Qual é o metal cujo símbolo químico é Au?",\n    "expected_answer": "Ouro",\n    "accepted_answers": [\n      "Ouro",\n      "Gold"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 21,\n    "question": "Qual é o resultado de 12 x 8?",\n    "expected_answer": "96",\n    "accepted_answers": [\n      "96"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 22,\n    "question": "Quanto é 144 dividido por 12?",\n    "expected_answer": "12",\n    "accepted_answers": [\n      "12"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 23,\n    "question": "Quanto é 15 ao quadrado?",\n    "expected_answer": "225",\n    "accepted_answers": [\n      "225"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 24,\n    "question": "Qual é a raiz quadrada de 81?",\n    "expected_answer": "9",\n    "accepted_answers": [\n      "9"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 25,\n    "question": "Quanto é 7 + 18?",\n    "expected_answer": "25",\n    "accepted_answers": [\n      "25"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 26,\n    "question": "Quanto é 100 - 37?",\n    "expected_answer": "63",\n    "accepted_answers": [\n      "63"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 27,\n    "question": "Quanto é 9 x 9?",\n    "expected_answer": "81",\n    "accepted_answers": [\n      "81"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 28,\n    "question": "Quanto é 5! (fatorial de 5)?",\n    "expected_answer": "120",\n    "accepted_answers": [\n      "120"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 29,\n    "question": "Qual é o próximo número na sequência 2, 4, 8, 16, ...?",\n    "expected_answer": "32",\n    "accepted_answers": [\n      "32"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 30,\n    "question": "Se um triângulo tem lados 3, 4 e 5, ele é retângulo?",\n    "expected_answer": "Sim",\n    "accepted_answers": [\n      "Sim",\n      "sim",\n      "Yes"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 31,\n    "question": "A água ferve a aproximadamente quantos graus Celsius ao nível do mar?",\n    "expected_answer": "100",\n    "accepted_answers": [\n      "100",\n      "100°C",\n      "100 graus Celsius"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 32,\n    "question": "Qual gás as plantas absorvem principalmente para fazer fotossíntese?",\n    "expected_answer": "Dióxido de carbono",\n    "accepted_answers": [\n      "Dióxido de carbono",\n      "CO2",\n      "Gás carbônico"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 33,\n    "question": "Qual é a fórmula química da água?",\n    "expected_answer": "H2O",\n    "accepted_answers": [\n      "H2O"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 34,\n    "question": "Qual partícula tem carga elétrica negativa?",\n    "expected_answer": "Elétron",\n    "accepted_answers": [\n      "Elétron",\n      "Eletron",\n      "Electron"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 35,\n    "question": "Qual órgão do corpo humano bombeia sangue?",\n    "expected_answer": "Coração",\n    "accepted_answers": [\n      "Coração",\n      "Coracao"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 36,\n    "question": "Qual é o satélite natural da Terra?",\n    "expected_answer": "Lua",\n    "accepted_answers": [\n      "Lua",\n      "Moon"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 37,\n    "question": "Qual é o processo pelo qual um líquido vira vapor?",\n    "expected_answer": "Evaporação",\n    "accepted_answers": [\n      "Evaporação",\n      "Evaporacao"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 38,\n    "question": "Quantos cromossomos tem um ser humano típico em células somáticas?",\n    "expected_answer": "46",\n    "accepted_answers": [\n      "46"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 39,\n    "question": "Qual é a camada da atmosfera onde vivemos?",\n    "expected_answer": "Troposfera",\n    "accepted_answers": [\n      "Troposfera"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 40,\n    "question": "O DNA tem forma clássica de qual estrutura?",\n    "expected_answer": "Dupla hélice",\n    "accepted_answers": [\n      "Dupla hélice",\n      "Dupla helix",\n      "Double helix"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 41,\n    "question": "Em que ano o homem pisou na Lua pela primeira vez?",\n    "expected_answer": "1969",\n    "accepted_answers": [\n      "1969"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 42,\n    "question": "Quem foi o primeiro presidente do Brasil?",\n    "expected_answer": "Deodoro da Fonseca",\n    "accepted_answers": [\n      "Deodoro da Fonseca"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 43,\n    "question": "Quem foi o líder sul-africano associado ao fim do apartheid e eleito presidente em 1994?",\n    "expected_answer": "Nelson Mandela",\n    "accepted_answers": [\n      "Nelson Mandela",\n      "Mandela"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 44,\n    "question": "Em que país começou a Revolução Industrial?",\n    "expected_answer": "Reino Unido",\n    "accepted_answers": [\n      "Reino Unido",\n      "Inglaterra",\n      "Grã-Bretanha",\n      "Great Britain"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 45,\n    "question": "Qual império construiu o Coliseu?",\n    "expected_answer": "Império Romano",\n    "accepted_answers": [\n      "Império Romano",\n      "Roman Empire",\n      "Roma"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 46,\n    "question": "Em que ano caiu o Muro de Berlim?",\n    "expected_answer": "1989",\n    "accepted_answers": [\n      "1989"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 47,\n    "question": "Quem liderou a independência da Índia com estratégia de não violência?",\n    "expected_answer": "Mahatma Gandhi",\n    "accepted_answers": [\n      "Mahatma Gandhi",\n      "Gandhi"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 48,\n    "question": "Qual civilização antiga construiu Machu Picchu?",\n    "expected_answer": "Inca",\n    "accepted_answers": [\n      "Inca",\n      "Império Inca"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 49,\n    "question": "Quem foi o imperador francês derrotado em Waterloo?",\n    "expected_answer": "Napoleão Bonaparte",\n    "accepted_answers": [\n      "Napoleão Bonaparte",\n      "Napoleon Bonaparte",\n      "Napoleão"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 50,\n    "question": "Em que continente fica o Egito?",\n    "expected_answer": "África",\n    "accepted_answers": [\n      "África",\n      "Africa"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 51,\n    "question": "Se todos os gatos são mamíferos e Mimi é um gato, Mimi é mamífero?",\n    "expected_answer": "Sim",\n    "accepted_answers": [\n      "Sim",\n      "sim",\n      "Yes"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 52,\n    "question": "Se hoje é segunda-feira, que dia será depois de 3 dias?",\n    "expected_answer": "Quinta-feira",\n    "accepted_answers": [\n      "Quinta-feira",\n      "Quinta",\n      "Thursday"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 53,\n    "question": "Qual palavra não pertence ao grupo: maçã, banana, cenoura, uva?",\n    "expected_answer": "Cenoura",\n    "accepted_answers": [\n      "Cenoura"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 54,\n    "question": "Se João é mais alto que Maria e Maria é mais alta que Ana, quem é a mais alta?",\n    "expected_answer": "João",\n    "accepted_answers": [\n      "João",\n      "Joao"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 55,\n    "question": "Um carro percorre 60 km em 1 hora. Em 3 horas, mantendo a velocidade, percorre quantos km?",\n    "expected_answer": "180",\n    "accepted_answers": [\n      "180"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 56,\n    "question": "Se 2 canetas custam 10 reais, quanto custam 5 canetas ao mesmo preço unitário?",\n    "expected_answer": "25",\n    "accepted_answers": [\n      "25",\n      "25 reais",\n      "R$25"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 57,\n    "question": "Qual número é ímpar: 14, 22, 31, 48?",\n    "expected_answer": "31",\n    "accepted_answers": [\n      "31"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 58,\n    "question": "Se A implica B e A é verdadeiro, o que se conclui sobre B?",\n    "expected_answer": "B é verdadeiro",\n    "accepted_answers": [\n      "B é verdadeiro",\n      "verdadeiro",\n      "B true"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 59,\n    "question": "Se uma receita pede 2 ovos para 1 bolo, quantos ovos são necessários para 4 bolos?",\n    "expected_answer": "8",\n    "accepted_answers": [\n      "8"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 60,\n    "question": "Qual alternativa completa melhor: livro está para leitura assim como garfo está para ...?",\n    "expected_answer": "Comer",\n    "accepted_answers": [\n      "Comer",\n      "alimentação",\n      "eat"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 61,\n    "question": "Em Python, qual palavra-chave define uma função?",\n    "expected_answer": "def",\n    "accepted_answers": [\n      "def"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 62,\n    "question": "Qual estrutura em Python normalmente guarda pares chave-valor?",\n    "expected_answer": "dicionário",\n    "accepted_answers": [\n      "dicionário",\n      "dicionario",\n      "dict"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 63,\n    "question": "Em JavaScript, qual valor booleano representa verdade?",\n    "expected_answer": "true",\n    "accepted_answers": [\n      "true"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 64,\n    "question": "Qual comando Git cria um novo commit com as mudanças staged?",\n    "expected_answer": "git commit",\n    "accepted_answers": [\n      "git commit"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 65,\n    "question": "Qual protocolo é usado normalmente para páginas web seguras?",\n    "expected_answer": "HTTPS",\n    "accepted_answers": [\n      "HTTPS"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 66,\n    "question": "Qual comando Linux lista arquivos de um diretório?",\n    "expected_answer": "ls",\n    "accepted_answers": [\n      "ls"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 67,\n    "question": "Em SQL, qual comando retorna linhas de uma tabela?",\n    "expected_answer": "SELECT",\n    "accepted_answers": [\n      "SELECT"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 68,\n    "question": "Em HTML, qual tag principal representa um link?",\n    "expected_answer": "<a>",\n    "accepted_answers": [\n      "<a>",\n      "a"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 69,\n    "question": "Em CSS, qual propriedade muda a cor do texto?",\n    "expected_answer": "color",\n    "accepted_answers": [\n      "color"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 70,\n    "question": "Qual estrutura de dados usa a política LIFO?",\n    "expected_answer": "pilha",\n    "accepted_answers": [\n      "pilha",\n      "stack"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 71,\n    "question": "Qual é o idioma oficial mais falado no Brasil?",\n    "expected_answer": "Português",\n    "accepted_answers": [\n      "Português",\n      "Portugues"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 72,\n    "question": "Qual é o antônimo de “rápido”?",\n    "expected_answer": "Lento",\n    "accepted_answers": [\n      "Lento"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 73,\n    "question": "Complete: cachorro está para latir assim como gato está para ...?",\n    "expected_answer": "miar",\n    "accepted_answers": [\n      "miar"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 74,\n    "question": "Qual é o plural de “papel” em português?",\n    "expected_answer": "papéis",\n    "accepted_answers": [\n      "papéis",\n      "papeis"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 75,\n    "question": "Qual palavra é um verbo: azul, correr, mesa, janela?",\n    "expected_answer": "correr",\n    "accepted_answers": [\n      "correr"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 76,\n    "question": "Qual é o sinônimo mais próximo de “feliz”?",\n    "expected_answer": "contente",\n    "accepted_answers": [\n      "contente",\n      "alegre"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 77,\n    "question": "Qual é a forma correta: “a gente fomos” ou “a gente foi”?",\n    "expected_answer": "a gente foi",\n    "accepted_answers": [\n      "a gente foi"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 78,\n    "question": "Em inglês, como se traduz “casa”?",\n    "expected_answer": "house",\n    "accepted_answers": [\n      "house"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 79,\n    "question": "Qual dessas palavras é um adjetivo: bonito, correr, pedra, ontem?",\n    "expected_answer": "bonito",\n    "accepted_answers": [\n      "bonito"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 80,\n    "question": "Qual palavra está escrita corretamente: exceção ou excessão?",\n    "expected_answer": "exceção",\n    "accepted_answers": [\n      "exceção",\n      "excecao"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 81,\n    "question": "Qual país venceu a Copa do Mundo de 2002?",\n    "expected_answer": "Brasil",\n    "accepted_answers": [\n      "Brasil",\n      "Brazil"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 82,\n    "question": "Quantos jogadores cada time tem em campo no futebol tradicional?",\n    "expected_answer": "11",\n    "accepted_answers": [\n      "11",\n      "onze"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 83,\n    "question": "Em que esporte se usa uma cesta e uma bola laranja?",\n    "expected_answer": "basquete",\n    "accepted_answers": [\n      "basquete",\n      "basketball"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 84,\n    "question": "Em qual esporte Rafael Nadal ficou mundialmente famoso?",\n    "expected_answer": "tênis",\n    "accepted_answers": [\n      "tênis",\n      "tenis",\n      "tennis"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 85,\n    "question": "Qual país sediou as Olimpíadas de 2016?",\n    "expected_answer": "Brasil",\n    "accepted_answers": [\n      "Brasil",\n      "Brazil"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 86,\n    "question": "Qual piloto brasileiro é tricampeão mundial de Fórmula 1 e morreu em 1994?",\n    "expected_answer": "Ayrton Senna",\n    "accepted_answers": [\n      "Ayrton Senna",\n      "Senna"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 87,\n    "question": "No xadrez, quantas casas tem um tabuleiro?",\n    "expected_answer": "64",\n    "accepted_answers": [\n      "64"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 88,\n    "question": "Qual esporte usa raquete, rede e peteca?",\n    "expected_answer": "badminton",\n    "accepted_answers": [\n      "badminton"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 89,\n    "question": "Em que esporte se faz touchdown?",\n    "expected_answer": "futebol americano",\n    "accepted_answers": [\n      "futebol americano",\n      "football americano",\n      "american football"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 90,\n    "question": "Qual seleção é conhecida como La Albiceleste?",\n    "expected_answer": "Argentina",\n    "accepted_answers": [\n      "Argentina"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 91,\n    "question": "Se você misturar azul com amarelo, qual cor obtém normalmente?",\n    "expected_answer": "verde",\n    "accepted_answers": [\n      "verde"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 92,\n    "question": "Qual eletrodoméstico é usado para manter alimentos congelados?",\n    "expected_answer": "freezer",\n    "accepted_answers": [\n      "freezer",\n      "congelador"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 93,\n    "question": "Quantos minutos tem 2 horas?",\n    "expected_answer": "120",\n    "accepted_answers": [\n      "120"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 94,\n    "question": "Qual objeto é usado para cortar papel com as mãos?",\n    "expected_answer": "tesoura",\n    "accepted_answers": [\n      "tesoura"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 95,\n    "question": "Qual é o mês que vem depois de abril?",\n    "expected_answer": "maio",\n    "accepted_answers": [\n      "maio"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 96,\n    "question": "Se um produto custa 50 reais e recebe desconto de 10 reais, quanto passa a custar?",\n    "expected_answer": "40",\n    "accepted_answers": [\n      "40",\n      "40 reais",\n      "R$40"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 97,\n    "question": "Qual bebida é tradicionalmente feita a partir de grãos de café torrados?",\n    "expected_answer": "café",\n    "accepted_answers": [\n      "café",\n      "cafe"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 98,\n    "question": "Qual direção é oposta ao norte?",\n    "expected_answer": "sul",\n    "accepted_answers": [\n      "sul",\n      "south"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 99,\n    "question": "Qual animal normalmente põe ovos e tem penas?",\n    "expected_answer": "ave",\n    "accepted_answers": [\n      "ave",\n      "pássaro",\n      "passaro",\n      "bird"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 100,\n    "question": "Quantos dias tem uma semana?",\n    "expected_answer": "7",\n    "accepted_answers": [\n      "7",\n      "sete"\n    ],\n    "category": "everyday",\n    "notes": ""\n  }\n]'
if not QUESTIONS_PATH.exists():
    QUESTIONS_PATH.write_text(QUESTIONS_PAYLOAD, encoding='utf-8')
    print('QUESTIONS_PATH criado:', QUESTIONS_PATH)

PACKAGE_CANDIDATES = [
    DEPLOY_DIR,
    EXPORTS_DIR / 'ternary_mlp_l78',
    PROJECT_ROOT / 'deploy',
    PROJECT_ROOT / 'exports' / 'ternary_mlp_l78',
]
PACKAGE_DIR = None
for candidate in PACKAGE_CANDIDATES:
    if (candidate / 'manifest.json').exists() and (candidate / 'mlp_ternary_2bit.bin').exists():
        PACKAGE_DIR = candidate
        break
if PACKAGE_DIR is None:
    checked = [str(p) for p in PACKAGE_CANDIDATES]
    raise FileNotFoundError('Pacote packed não encontrado em: ' + json.dumps(checked, ensure_ascii=False, indent=2))

MANIFEST_PATH = PACKAGE_DIR / 'manifest.json'
BIN_PATH = PACKAGE_DIR / 'mlp_ternary_2bit.bin'

RUN_FUNCTIONAL_RECONSTRUCTION_EVAL = True
MAX_NEW_TOKENS = 24
BATCH_SIZE = 1
PROMPT_TEMPLATE = 'Responda de forma curta e direta.\nPergunta: {question}\nResposta:'

print('PRUNED_CHECKPOINT', PRUNED_CHECKPOINT.exists(), PRUNED_CHECKPOINT)
print('L78_ADAPTER', L78_ADAPTER.exists(), L78_ADAPTER)
print('QUESTIONS_PATH', QUESTIONS_PATH.exists(), QUESTIONS_PATH)
print('PACKAGE_DIR', PACKAGE_DIR)
print('MANIFEST', MANIFEST_PATH.exists(), MANIFEST_PATH)
print('BIN', BIN_PATH.exists(), BIN_PATH, BIN_PATH.stat().st_size if BIN_PATH.exists() else None)


In [ ]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')


def human_bytes(n):
    units = ['B', 'KB', 'MB', 'GB', 'TB']
    value = float(n)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f'{value:.2f} {unit}'
        value /= 1024


def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

manifest = load_json(MANIFEST_PATH)
questions = load_json(QUESTIONS_PATH)

assert manifest.get('format') == 'rpt_hybrid_ternary_mlp_lora_v1', manifest.get('format')
assert manifest['summary']['num_tensors'] == 78, manifest['summary']
assert BIN_PATH.exists(), BIN_PATH
assert BIN_PATH.stat().st_size == manifest['summary']['packed_total_bytes'], (BIN_PATH.stat().st_size, manifest['summary']['packed_total_bytes'])

print(json.dumps(manifest['summary'], ensure_ascii=False, indent=2))
print('bin_sha256 =', sha256_file(BIN_PATH))


In [ ]:
# Funções 2-bit: codebook 0=zero, 1=+scale, 2=-scale, 3=reserved.
def unpack_codes_from_blob(blob, n):
    raw = np.frombuffer(blob, dtype=np.uint8)
    codes = np.empty(raw.size * 4, dtype=np.uint8)
    codes[0::4] = raw & 0b11
    codes[1::4] = (raw >> 2) & 0b11
    codes[2::4] = (raw >> 4) & 0b11
    codes[3::4] = (raw >> 6) & 0b11
    return codes[:n]


def unpack_tensor_from_blob(blob, n, shape, scale, dtype=torch.float32):
    codes = unpack_codes_from_blob(blob, n)
    vals = np.zeros(n, dtype=np.float32)
    vals[codes == 1] = scale
    vals[codes == 2] = -scale
    tensor = torch.from_numpy(vals).view(tuple(shape))
    return tensor.to(dtype=dtype)


def tensor_code_hist_from_blob(blob, n):
    codes = unpack_codes_from_blob(blob, n)
    counts = np.bincount(codes, minlength=4).astype(np.int64)
    return {str(i): int(counts[i]) for i in range(4)}

# Validação estrutural completa, sem carregar modelo.
tensors = manifest['tensors']
structural_errors = []
structural_rows = []
expected_offset = 0
code_totals = np.zeros(4, dtype=np.int64)
selected_indexes = sorted(set([0, 1, 2, len(tensors)//2, len(tensors)-3, len(tensors)-2, len(tensors)-1]))
selected_reconstruction_checks = []

with open(BIN_PATH, 'rb') as f:
    for idx, item in enumerate(tensors):
        if item['offset'] != expected_offset:
            structural_errors.append({'name': item['name'], 'error': 'non_contiguous_offset', 'expected': expected_offset, 'actual': item['offset']})
        expected_offset = item['offset'] + item['packed_bytes']

        expected_packed = (item['numel'] + 3) // 4
        if item['packed_bytes'] != expected_packed:
            structural_errors.append({'name': item['name'], 'error': 'packed_size_mismatch', 'expected': expected_packed, 'actual': item['packed_bytes']})

        f.seek(item['offset'])
        blob = f.read(item['packed_bytes'])
        if len(blob) != item['packed_bytes']:
            structural_errors.append({'name': item['name'], 'error': 'short_read', 'expected': item['packed_bytes'], 'actual': len(blob)})
            continue

        hist = tensor_code_hist_from_blob(blob, item['numel'])
        counts = np.array([hist[str(i)] for i in range(4)], dtype=np.int64)
        code_totals += counts
        reserved_ratio = counts[3] / max(1, item['numel'])
        if counts[3] != 0:
            structural_errors.append({'name': item['name'], 'error': 'reserved_code_present', 'reserved_count': int(counts[3])})

        structural_rows.append({
            'name': item['name'],
            'shape': item['shape'],
            'numel': item['numel'],
            'scale': item['scale'],
            'packed_bytes': item['packed_bytes'],
            'fp16_bytes': item['fp16_bytes'],
            'compression_vs_fp16': item['compression_vs_fp16'],
            'code_histogram': hist,
            'zero_ratio': round(counts[0] / item['numel'], 8),
            'positive_ratio': round(counts[1] / item['numel'], 8),
            'negative_ratio': round(counts[2] / item['numel'], 8),
            'reserved_ratio': round(reserved_ratio, 8),
        })

        if idx in selected_indexes:
            recon = unpack_tensor_from_blob(blob, item['numel'], item['shape'], item['scale'])
            uniq = sorted(float(x) for x in torch.unique(recon).tolist())
            selected_reconstruction_checks.append({
                'name': item['name'],
                'shape': item['shape'],
                'unique_values_count': len(uniq),
                'unique_values': uniq,
                'abs_max': float(recon.abs().max().item()),
                'mean_abs': float(recon.abs().mean().item()),
            })
            del recon

structural_summary = {
    'manifest_path': str(MANIFEST_PATH),
    'bin_path': str(BIN_PATH),
    'bin_size_bytes': BIN_PATH.stat().st_size,
    'bin_size_human': human_bytes(BIN_PATH.stat().st_size),
    'bin_sha256': sha256_file(BIN_PATH),
    'num_tensors': len(tensors),
    'total_numel': int(sum(x['numel'] for x in tensors)),
    'expected_final_offset': expected_offset,
    'offset_matches_file_size': expected_offset == BIN_PATH.stat().st_size,
    'manifest_summary': manifest['summary'],
    'code_totals': {str(i): int(code_totals[i]) for i in range(4)},
    'global_zero_ratio': round(float(code_totals[0]) / max(1, int(code_totals.sum())), 8),
    'global_positive_ratio': round(float(code_totals[1]) / max(1, int(code_totals.sum())), 8),
    'global_negative_ratio': round(float(code_totals[2]) / max(1, int(code_totals.sum())), 8),
    'global_reserved_ratio': round(float(code_totals[3]) / max(1, int(code_totals.sum())), 8),
    'structural_errors': structural_errors,
    'selected_reconstruction_checks': selected_reconstruction_checks,
}

structural_report = {
    'status': 'passed' if not structural_errors and structural_summary['offset_matches_file_size'] else 'failed',
    'summary': structural_summary,
    'tensor_rows': structural_rows,
}
structural_report_path = AUDIT_DIR / 'packed_ternary_reconstruction_structural.json'
save_json(structural_report, structural_report_path)
print('saved', structural_report_path)
print(json.dumps(structural_summary, ensure_ascii=False, indent=2)[:8000])
if structural_report['status'] != 'passed':
    raise RuntimeError('Structural packed validation failed; inspect structural_errors.')


In [ ]:
# Canonical evaluator: use a checked-out RPT revision, never an embedded copy.
import os, sys
from pathlib import Path
_rpt_candidates = [Path(os.environ.get('RPT_SOURCE_ROOT', '.')).expanduser().resolve(), Path('/content/RPT')]
_rpt_candidates += [Path.cwd(), *Path.cwd().parents]
if 'PROJECT_ROOT' in globals():
    _rpt_candidates += [Path(PROJECT_ROOT), Path(PROJECT_ROOT).parent]
_rpt_source = next((p for p in _rpt_candidates if (p / 'validation' / 'evaluation.py').is_file()), None)
if _rpt_source is None:
    raise RuntimeError('Clone a fixed RPT revision and set RPT_SOURCE_ROOT to that checkout; see REPRODUCIBILITY.md.')
if str(_rpt_source) not in sys.path:
    sys.path.insert(0, str(_rpt_source))
from validation.evaluation import EVALUATOR_VERSION, answer_is_correct
print('Evaluator:', EVALUATOR_VERSION, '| suite classification: development')

# Avaliação local: mesma lógica dos notebooks anteriores.
def normalize_text(s):
    return re.sub(r'\s+', ' ', str(s).strip().lower())


def judge_answer(question_payload, answer):
    accepted = [question_payload.get('expected_answer', '')]
    accepted.extend(question_payload.get('accepted_answers') or [])
    return answer_is_correct(answer, accepted)


def extract_answer(prompt, generated):
    text = generated
    if prompt in text:
        text = text.split(prompt, 1)[-1]
    text = text.strip()
    text = text.split('\n')[0].strip()
    return text


def run_eval(model, tokenizer, questions, batch_size=1, max_new_tokens=24, verbose=True):
    rows = []
    correct = 0
    device = next(model.parameters()).device
    for idx, q in enumerate(questions):
        prompt = PROMPT_TEMPLATE.format(question=q['question'])
        inputs = tokenizer(prompt, return_tensors='pt').to(device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        generated = tokenizer.decode(out[0], skip_special_tokens=True)
        answer = extract_answer(prompt, generated)
        ok = judge_answer(q, answer)
        correct += int(ok)
        row = {
            'id': q.get('id', idx + 1),
            'category': q.get('category'),
            'question': q['question'],
            'expected_answer': q['expected_answer'],
            'model_answer': answer,
            'is_correct': ok,
            'raw_output': generated,
        }
        rows.append(row)
        if verbose:
            print(f"{idx+1:03d}/{len(questions)}", 'OK' if ok else 'ERR', '|', q['question'], '=>', answer)
    by_category = {}
    for row in rows:
        cat = row.get('category') or 'unknown'
        by_category.setdefault(cat, {'total': 0, 'correct': 0})
        by_category[cat]['total'] += 1
        by_category[cat]['correct'] += int(row['is_correct'])
    for cat, vals in by_category.items():
        vals['accuracy'] = vals['correct'] / vals['total'] if vals['total'] else 0.0
    summary = {
        'total_questions': len(questions),
        'correct': correct,
        'accuracy': correct / len(questions) if questions else 0.0,
        'by_category': by_category,
    }
    return rows, summary


In [ ]:
# Reconstrói MLPs packed dentro do student_pruned e aplica o adapter l78.
# Esta é a prova funcional do pacote packed reconstruído.
functional_result = None
if RUN_FUNCTIONAL_RECONSTRUCTION_EVAL:
    cleanup_memory()
    if not PRUNED_CHECKPOINT.exists():
        raise FileNotFoundError(f'PRUNED_CHECKPOINT ausente: {PRUNED_CHECKPOINT}')
    if not L78_ADAPTER.exists():
        raise FileNotFoundError(f'L78_ADAPTER ausente: {L78_ADAPTER}')

    tokenizer = AutoTokenizer.from_pretrained(str(PRUNED_CHECKPOINT), token=HF_TOKEN)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        str(PRUNED_CHECKPOINT),
        dtype='auto',
        device_map='auto',
        token=HF_TOKEN,
    )
    model.eval()

    named_params = dict(model.named_parameters())
    missing_params = []
    meta_params = []
    applied = []
    start = time.time()

    with open(BIN_PATH, 'rb') as f:
        for item in manifest['tensors']:
            name = item['name']
            param = named_params.get(name)
            if param is None:
                missing_params.append(name)
                continue
            if getattr(param, 'is_meta', False):
                meta_params.append(name)
                continue
            f.seek(item['offset'])
            blob = f.read(item['packed_bytes'])
            recon = unpack_tensor_from_blob(blob, item['numel'], item['shape'], item['scale'], dtype=torch.float32)
            if tuple(param.shape) != tuple(recon.shape):
                raise RuntimeError(f'Shape mismatch for {name}: param={tuple(param.shape)} recon={tuple(recon.shape)}')
            with torch.no_grad():
                param.copy_(recon.to(device=param.device, dtype=param.dtype))
            applied.append(name)
            del recon

    if missing_params or meta_params:
        raise RuntimeError(json.dumps({'missing_params': missing_params, 'meta_params': meta_params}, ensure_ascii=False, indent=2))
    if len(applied) != 78:
        raise RuntimeError(f'Expected 78 packed MLP tensors applied, got {len(applied)}')

    adapter_model = PeftModel.from_pretrained(model, str(L78_ADAPTER), token=HF_TOKEN, is_trainable=False)
    adapter_model.eval()

    rows, eval_summary = run_eval(adapter_model, tokenizer, questions, batch_size=BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS, verbose=True)
    elapsed = round(time.time() - start, 2)

    functional_result = {
        'status': 'completed',
        'source_pruned_checkpoint': str(PRUNED_CHECKPOINT),
        'adapter': str(L78_ADAPTER),
        'packed_manifest': str(MANIFEST_PATH),
        'packed_bin': str(BIN_PATH),
        'applied_packed_mlp_tensors': len(applied),
        'elapsed_seconds_after_model_load_start': elapsed,
        'summary': eval_summary,
        'rows': rows,
        'note': 'This evaluates reconstructed packed MLP tensors injected into student_pruned plus the l78 PEFT adapter. It still materializes tensors into torch weights; it is not an efficient 2-bit runtime kernel.',
    }
    functional_path = AUDIT_DIR / 'packed_ternary_reconstruction_functional_eval.json'
    save_json(functional_result, functional_path)
    print('saved', functional_path)
    print(json.dumps(eval_summary, ensure_ascii=False, indent=2))

    del adapter_model, model
    cleanup_memory()
else:
    print('RUN_FUNCTIONAL_RECONSTRUCTION_EVAL=False; pulando avaliação funcional.')


In [ ]:
# Relatório final consolidado e zip de artefatos.
EXPECTED_MIN_FUNCTIONAL_ACCURACY = 0.90
functional_summary = functional_result.get('summary') if functional_result else None
functional_accuracy = functional_summary.get('accuracy') if functional_summary else None
functional_passed = functional_accuracy is not None and functional_accuracy >= EXPECTED_MIN_FUNCTIONAL_ACCURACY

final_report = {
    'status': 'completed' if structural_report['status'] == 'passed' and functional_passed else 'functional_regression_detected',
    'what_this_proves': 'The packed 2-bit MLP artifact can be structurally decoded and materialized back into torch tensors.',
    'what_this_does_not_prove': 'It does not prove efficient inference directly over the packed 2-bit representation. If functional accuracy is below target, it also does not prove that the current packed package reproduces the recovered L78 quality.',
    'structural_report_path': str(AUDIT_DIR / 'packed_ternary_reconstruction_structural.json'),
    'functional_report_path': str(AUDIT_DIR / 'packed_ternary_reconstruction_functional_eval.json') if functional_result else None,
    'structural_summary': structural_summary,
    'functional_summary': functional_summary,
    'expected_reference_accuracy': {
        'local_l78_previous': 0.92,
        'external_scaled500_delta_range': '[-0.006, +0.006] vs student_pruned_base',
        'minimum_expected_functional_accuracy': EXPECTED_MIN_FUNCTIONAL_ACCURACY,
    },
    'interpretation': {
        'functional_passed': functional_passed,
        'functional_accuracy': functional_accuracy,
        'warning': 'A low functional accuracy means the packed/materialized ternary base plus adapter does not match the prior clean-base PEFT L78 result.',
    },
}
final_report_path = AUDIT_DIR / 'packed_ternary_reconstruction_report.json'
save_json(final_report, final_report_path)
print('saved', final_report_path)
print(json.dumps(final_report, ensure_ascii=False, indent=2)[:10000])

artifact_paths = [
    AUDIT_DIR / 'packed_ternary_reconstruction_structural.json',
    AUDIT_DIR / 'packed_ternary_reconstruction_report.json',
]
if functional_result:
    artifact_paths.append(AUDIT_DIR / 'packed_ternary_reconstruction_functional_eval.json')

zip_path = AUDIT_DIR / 'packed_ternary_reconstruction_artifacts.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in artifact_paths:
        if path.exists():
            zf.write(path, arcname=path.name)
print('ZIP salvo em:', zip_path)

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
